In [ ]:
import os

from modules.context import get_concepts

from pathlib import Path
os.chdir(Path.cwd().parent)
import time
import copy
import json
import random
import warnings
import numpy as np
from pathlib import Path
from collections import defaultdict
from IPython.core.magic import register_cell_magic

from data.dataloader import load_dataset
from utils.utils import set_random_seed, Usage
from utils.llm import load_model
from utils.metrics import get_metrics_fn
from utils.multiprocess import multiprocess_call
from modules.post_process import get_label_tokenizer, get_logit, prepare_labels, match_condition_fuzz
from TherapyReason.significance import get_significance
from TherapyReason.therapy import get_dl_candidates, get_kg_candidates, get_therapy, rank_dl_candidates
from TherapyReason.overview import process_solution_overview_wrap
from TherapyReason.reasoning import candidate2string, template_solution_dl_diag, solution2string_with_overview, template_solution_system_diag, \
    post_process_solution_wrapper_diag, template_solution_kg_diag, candidate_similarity

set_random_seed(42)
dataset = 'mimic4'
task = 'diagnosis'
label_key = 'label'
datasets, full_set = load_dataset(dataset, task)
test_set = datasets['test']
label_tokenizer = get_label_tokenizer(full_set, label_key)
cond_tokenizer = get_label_tokenizer(full_set, 'conditions')
proc_tokenizer = get_label_tokenizer(full_set, 'procedures')
drug_tokenizer = get_label_tokenizer(full_set, 'drugs')
significance = get_significance(dataset, task)
cache_dir = f"./TherapyReason/cache/{dataset}-{task}"
os.makedirs(cache_dir, exist_ok=True)
concept_similarity = json.loads(Path("TherapyReason/cache/llm_pairwise.json").read_text())
print(len(test_set))

warnings.filterwarnings('ignore')


@register_cell_magic
def skip(line, cell): return None


llm = load_model({"model": "DeepSeek-V3-671B", "server": "ant", "do_sample": False, "temperature": 0.0, "max_attempts": 10})

In [ ]:
from modules.candidate import load_diagnosis_candidate_model
from tqdm import tqdm
import torch

def get_dl_candidates(thres, cache_dir, dataset, task, full_set, test_set, model_name):
    assert task in ['drugrec', 'diagnosis']
    label_key = 'drugs' if task == 'drugrec' else 'label'
    load_model_fn = load_diagnosis_candidate_model

    dl_candidate_file = Path(os.path.join(cache_dir, f"dl_{model_name}_thres{int(thres * 10)}.jsonl"))
    if dl_candidate_file.exists():
        sample_dl_candidates = [json.loads(l) for l in dl_candidate_file.read_text().splitlines()]
    else:
        sample_dl_candidates = []
        dl_model = load_model_fn(dataset, task, full_set, model_name)
        for sample in tqdm(test_set):
            conditions, procedures, drugs = get_concepts(sample, task, with_history=True, flatten=False)
            kwargs = {'conditions': [conditions], 'procedures': [procedures], 'drugs': [drugs]}
            kwargs['patient_id'] = [sample['patient_id']]
            kwargs['visit_id'] = [sample['visit_id']]
            kwargs[label_key] = [sample[label_key]]
            with torch.no_grad():
                output = dl_model(**kwargs)

            solution_score = output['y_prob'][0].cpu().numpy()  # (n_drugs_total,)
            thresholded_indices = np.where(solution_score > thres)[0]
            candidates = dl_model.label_tokenizer.convert_indices_to_tokens(thresholded_indices)
            sample_dl_candidates.append(candidates)

        dl_candidate_file.write_text("\n".join(json.dumps(x) for x in sample_dl_candidates))

    print(np.mean([len(x) for x in sample_dl_candidates]))
    return sample_dl_candidates

def rank_dl_candidates(cache_dir, dataset, task, full_set, test_set, model_name):
    assert task in ['drugrec', 'diagnosis']
    label_key = 'drugs' if task == 'drugrec' else 'label'
    load_model_fn = load_diagnosis_candidate_model

    dl_ranked_file = Path(os.path.join(cache_dir, f"dl_{model_name}_rank.jsonl"))
    if dl_ranked_file.exists():
        sample_dl_ranks = [json.loads(l) for l in dl_ranked_file.read_text().splitlines()]
    else:
        sample_dl_ranks = []
        dl_model = load_model_fn(dataset, task, full_set, model_name)
        for sample in tqdm(test_set):
            conditions, procedures, drugs = get_concepts(sample, task, with_history=True, flatten=False)
            kwargs = {'conditions': [conditions], 'procedures': [procedures], 'drugs': [drugs]}
            kwargs['patient_id'] = [sample['patient_id']]
            kwargs['visit_id'] = [sample['visit_id']]
            kwargs[label_key] = [sample[label_key]]
            with torch.no_grad():
                output = dl_model(**kwargs)

            solution_score = output['y_prob'][0].cpu().numpy()
            sorted_indices = np.argsort(-solution_score)
            all_candidates = dl_model.label_tokenizer.convert_indices_to_tokens(sorted_indices)
            sample_dl_ranks.append(all_candidates)

            dl_ranked_file.write_text("\n".join(json.dumps(x) for x in sample_dl_ranks))

    print(np.mean([len(x) for x in sample_dl_ranks]))
    return sample_dl_ranks

In [ ]:
# Discovery
thres = 0.4
model_name = 'TRANS'
sample_dl_candidates = get_dl_candidates(thres, os.path.join(cache_dir, "backbone"), dataset, task, full_set, test_set, model_name)
sample_candidate_ranks = rank_dl_candidates(os.path.join(cache_dir, "backbone"), dataset, task, full_set, test_set, model_name)
sample_kg_candidates = get_kg_candidates(cache_dir, task, test_set, label_tokenizer, label_tokenizer)

In [ ]:
# Therapy (or Solution)
solution_file = Path(os.path.join(cache_dir, "backbone", f"solution_{model_name}_thres{int(thres * 10)}.jsonl"))
sample_solutions = get_therapy(solution_file, task, significance, label_key, test_set, sample_dl_candidates, sample_kg_candidates)

In [ ]:
# solution overview
process_solution_overview = process_solution_overview_wrap(
    os.path.join(cache_dir, "backbone", f"solution_{model_name}_thres{int(thres * 10)}_overview", llm.genre), llm)
solution_overview_output = multiprocess_call(process_solution_overview, sample_solutions, n_workers=16)

## solution reasoning

In [ ]:
# 初始化solution reasoning的cache文件（写空文件）
n_shot = 2
chunk_size = 20
top_p_kg = 60

for key, item in enumerate(list(zip(sample_solutions, solution_overview_output))):
    solution_obj, solution_overview = item
    visits = solution_obj['solutions']

    cache_dl_file = Path(os.path.join(cache_dir, "backbone", f"solution_{model_name}_thres{int(thres * 10)}_dl", llm.genre, f"{key}.json"))
    os.makedirs(os.path.dirname(cache_dl_file), exist_ok=True)
    if not cache_dl_file.exists():
        cached = []
        for visit_id, visit in enumerate(visits):
            _cached = []
            for solution_id, solution in enumerate(visit):
                n_chunks = int(np.ceil(len(solution["dl_candidates"]) / chunk_size))
                _cached.append({"dl_candidates": solution["dl_candidates"], "dl_query": [None] * n_chunks, "dl_response": [None] * n_chunks,
                                "dl_prediction": {}, "dl_usage": Usage().dict()})
            cached.append(_cached)
        cache_dl_file.write_text(json.dumps(cached, indent=4), encoding="utf-8")
        print(f"Init {cache_dl_file}")

    cache_kg_file = Path(os.path.join(cache_dir, "backbone", f"solution_{model_name}_thres{int(thres * 10)}_kg_{n_shot}shot", llm.genre, f"{key}.json"))
    os.makedirs(os.path.dirname(cache_kg_file), exist_ok=True)
    if not cache_kg_file.exists():
        cached = []
        for visit_id, visit in enumerate(visits):
            _cached = []
            for solution_id, solution in enumerate(visit):
                n_chunks = int(np.ceil(len(solution["kg_candidates"]) / chunk_size))
                _cached.append({"kg_candidates": solution["kg_candidates"], "kg_query": [None] * n_chunks, "kg_response": [None] * n_chunks,
                                "kg_prediction": {},"kg_usage": Usage().dict()})
            cached.append(_cached)

        cache_kg_file.write_text(json.dumps(cached, indent=4), encoding="utf-8")
        print(f"Init {cache_kg_file}")

In [ ]:
# 对DL进行reasoning
def process_solutions_dl(key, item, **kwargs):
    solution_obj, solution_overview = item
    visits = solution_obj['solutions']

    cache_file = Path(os.path.join(cache_dir, "backbone", f"solution_{model_name}_thres{int(thres * 10)}_dl", llm.genre, f"{key}.json"))
    cached = json.loads(cache_file.read_text(encoding="utf-8"))
    for visit_id, visit in enumerate(visits):
        for solution_id, solution in enumerate(visit):
            overview = solution_overview[visit_id][solution_id]['overview']

            n_chunks = int(np.ceil(len(solution['dl_candidates']) / chunk_size).item())
            for i in range(n_chunks):
                if cached[visit_id][solution_id]['dl_response'][i] is not None: continue

                time.sleep(random.randint(0, 4))
                dl_candidates = solution["dl_candidates"][i * chunk_size: (i + 1) * chunk_size]
                candidate_string_dict = candidate2string(dl_candidates, solution)

                post_process_solution = post_process_solution_wrapper_diag(label_tokenizer, candidates=dl_candidates)
                query = template_solution_dl_diag.format(solution_string=solution2string_with_overview(solution, overview, overview_only=True),
                                                         candidate_dict=json.dumps(candidate_string_dict, indent=2))
                _, (response, prediction), _, usage = llm.understand(query, system=template_solution_system_diag, post_process=post_process_solution,
                                                                     **kwargs)
                print(f"Patient: {key}\tVisit: {visit_id + 1}/{len(visits)}\tSolution: {solution_id + 1}/{len(visit)}\tDL Chunk: {i}/{n_chunks}")
                cached[visit_id][solution_id]['dl_query'][i] = query
                cached[visit_id][solution_id]['dl_response'][i] = response
                cached[visit_id][solution_id]['dl_prediction'].update(prediction)
                cached[visit_id][solution_id]['dl_usage'] = (usage + cached[visit_id][solution_id]['dl_usage']).dict()
                cache_file.write_text(json.dumps(cached, indent=4), encoding="utf-8")

    return key, cached


sample_dl_reasoning = multiprocess_call(process_solutions_dl, list(zip(sample_solutions, solution_overview_output)), n_workers=16, verbose=False)

In [ ]:
# 对KG进行reasoning
def process_solutions_kg(key, item, **kwargs):
    solution_obj, solution_overview, candidate_ranks = item
    visits = solution_obj['solutions']

    cache_dl_file = Path(os.path.join(cache_dir, "backbone", f"solution_{model_name}_thres{int(thres * 10)}_dl", llm.genre, f"{key}.json"))
    cache_file = Path(os.path.join(cache_dir, "backbone", f"solution_{model_name}_thres{int(thres * 10)}_kg_{n_shot}shot", llm.genre, f"{key}.json"))
    dl_cached = json.loads(cache_dl_file.read_text(encoding="utf-8"))
    cached = json.loads(cache_file.read_text(encoding="utf-8"))
    for visit_id, visit in enumerate(visits):
        for solution_id, solution in enumerate(visit):
            overview = solution_overview[visit_id][solution_id]['overview']

            # 将kg drug降序排序
            solution['kg_candidates'] = [x for x in candidate_ranks if x in solution['kg_candidates']]
            solution['kg_candidates'] = solution['kg_candidates'][:top_p_kg]

            n_chunks = int(np.ceil(len(solution['kg_candidates']) / chunk_size).item())
            for i in range(n_chunks):
                if cached[visit_id][solution_id]['kg_response'][i] is not None: continue

                time.sleep(random.randint(0, 4))
                kg_candidates = solution["kg_candidates"][i * chunk_size: (i + 1) * chunk_size]
                candidate_string_dict = candidate2string(kg_candidates, solution)
                assert candidate_string_dict

                # Use top-5 most similar DL drugs as few-shot examples
                dl_prediction = dl_cached[visit_id][solution_id]['dl_prediction']
                fewshot_similarities = [(key, candidate_similarity(key, kg_candidates, concept_similarity)) for key, value in dl_prediction.items()]
                fewshot_similarity_sorted = sorted(fewshot_similarities, key=lambda x: x[1], reverse=True)
                fewshots = {key: dl_prediction[key] for key, score in fewshot_similarity_sorted[:n_shot]}

                post_process_solution = post_process_solution_wrapper_diag(drug_tokenizer, candidates=kg_candidates)
                query = template_solution_kg_diag.format(solution_string=solution2string_with_overview(solution, overview, overview_only=True),
                                                    candidate_dict=json.dumps(candidate_string_dict, indent=2),
                                                    few_shot_dict=json.dumps(fewshots, indent=2))
                _, (response, prediction), _, usage = llm.understand(query, system=template_solution_system_diag, post_process=post_process_solution,
                                                                     **kwargs)
                print(f"Patient: {key}\tVisit: {visit_id + 1}/{len(visits)}\tSolution: {solution_id + 1}/{len(visit)}\tKG Chunk: {i}/{n_chunks}")
                cached[visit_id][solution_id]['kg_query'][i] = query
                cached[visit_id][solution_id]['kg_response'][i] = response
                cached[visit_id][solution_id]['kg_prediction'].update(prediction)
                cached[visit_id][solution_id]['kg_usage'] = (usage + cached[visit_id][solution_id]['kg_usage']).dict()
                cache_file.write_text(json.dumps(cached, indent=4), encoding="utf-8")

    return key, cached


sample_kg_reasoning = multiprocess_call(process_solutions_kg, list(zip(sample_solutions, solution_overview_output, sample_candidate_ranks)), n_workers=16, verbose=False)

# eval

In [ ]:
# 评估DL+KG的结果
def get_dlkg_metric(offset, threshold, dl_reasoning, kg_reasoning, agg='mean'):
    y_true_all, y_prob_all = [], []
    for i in range(len(test_set)):
        patient_prediction = defaultdict(list)
        for visit_id, visit in enumerate(dl_reasoning[i]):
            for solution_id in range(len(visit)):
                for entity, value in dl_reasoning[i][visit_id][solution_id]['dl_prediction'].items():
                    patient_prediction[entity].append(value['predict'] + offset)
                for entity, value in kg_reasoning[i][visit_id][solution_id]['kg_prediction'].items():
                    patient_prediction[entity].append(value['predict'])

        patient_prediction = {entity: np.mean(scores) if agg == 'mean' else np.max(scores) for entity, scores in patient_prediction.items()}
        patient_prediction = [entity for entity, score in patient_prediction.items() if score >= threshold]
        y_true_all.append(prepare_labels(task, [test_set[i][label_key]], label_tokenizer)[0].numpy())
        y_prob_all.append(get_logit(task, list(patient_prediction), label_tokenizer))

    metrics = get_metrics_fn(task)(np.stack(y_true_all), np.array(y_prob_all))
    print(offset, threshold, agg, metrics["precision_samples"], metrics["recall_samples"], metrics["f1_samples"], metrics["jaccard_samples"], metrics['hamming_loss'], metrics['rank'])

for agg in ['mean', 'max']:
    for offset in range(2, 4):
        for threshold in range(offset + 1, min(offset + 5, 5)):
            get_dlkg_metric(offset, threshold, sample_dl_reasoning, sample_kg_reasoning, agg=agg)